# Finding Redundant 1D Subspaces in LLM Activations

This notebook discovers **redundant 1-dimensional subspaces** in language model activations: directions where:
1. Activations have **large projections** (model actively uses this direction)
2. **Removing** this projection has **minimal impact** on output token distributions

In [1]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [2]:
from scripts.utils.load_dataset import DatasetName, load_dataset

ds = load_dataset(DatasetName.AYA_DATASET)


Materializing Dataset:   0%|          | 0/204000 [00:00<?, ?it/s]

INFO 03-08 17:14:30 [scripts.utils.load_dataset:272] Saving dataset to disk at /home/fre.gilad/source/silent_direction/scripts/utils/../../data/aya-dataset


Saving the dataset (0/1 shards):   0%|          | 0/20000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2500 [00:00<?, ? examples/s]

In [3]:
ds = load_dataset(DatasetName.TULU_V3)

Materializing Dataset:   0%|          | 0/939000 [00:00<?, ?it/s]

INFO 03-08 17:17:05 [scripts.utils.load_dataset:156] Saving dataset to disk at /home/fre.gilad/source/silent_direction/scripts/utils/../../data/tulu-v3


Saving the dataset (0/1 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
import torch
from src.utils.env import set_seed

set_seed(42)

# torch.set_float32_matmul_precision("high")

In [ ]:
import torch
from scripts.utils.load_model import load_model
from src.model import TargetedModel
from src.aliases import Conv

model_name = "microsoft/Phi-3-mini-4k-instruct"

print(f"====== Testing model {model_name} ======")
model, tokenizer = load_model(model_name)
targeted_model = TargetedModel(model, tokenizer, is_chat=True)
print(f"Model {model_name}")
print(model)

In [ ]:
print(model.dtype)

In [ ]:
from scripts.utils.load_dataset import load_dataset
from src.data import TableLoader

ds_train, ds_val, ds_test = load_dataset("hh-rlhf")

dl_train = TableLoader(ds_train, batch_size=16, shuffle=True)
dl_val = TableLoader(ds_val, batch_size=8, shuffle=True)
dl_test = TableLoader(ds_test, batch_size=8, shuffle=True)

In [ ]:
math_prompt_requires_reasoning = "What is the integral of the following complex function: f(x) = e^(x^2) * sin(x)?"

In [ ]:
response = targeted_model.generate(
    prompts=[[{"role": "user", "content": math_prompt_requires_reasoning}]]
)

In [ ]:
print(response)